In [ ]:
"""
MicroLive Notebook
==================
This notebook requires MicroLive to be installed:
    pip install microlive

For development mode:
    pip install -e /path/to/microlive
"""
# MicroLive imports
from microlive import microscopy as mi
from microlive.utils.device import check_gpu_status

# Verify GPU support
check_gpu_status()

# Standard scientific imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
def TASEP_SSA(k, t_array, timePerturbationApplication=0, evaluatingInhibitor=0, evaluatingFRAP=0):
    """
    Simulates the stochastic occupancy of ribosomes along an mRNA using the SSA (Stochastic Simulation Algorithm).

    Parameters:
    - k: numpy array of kinetic rates.
         The first element is the initiation rate (k_bind),
         the last element is the termination rate (k_termination),
         and the elements in between are the elongation rates (k_elongation).
    - t_array: numpy array of time points at which to record the system's state.
    - timePerturbationApplication: float, the time at which an inhibitor or perturbation is applied.
    - evaluatingInhibitor: int (1 or 0), flag indicating whether to simulate the effect of an inhibitor.
    - evaluatingFRAP: int (1 or 0), flag indicating whether to simulate a FRAP experiment.

    Returns:
    - ribosome_trajectories: numpy array where each row represents the trajectory of a ribosome.
                             The columns correspond to time points in t_array.
    - occupancy_output: numpy array containing the occupancy vectors over time.
                        Each column corresponds to a time point in t_array.
    """
    ## Defining parameters
    exclusion = 9                       # Exclusion volume (ribosome footprint)
    k_bind = k[0]                       # Initiation rate
    k_elongation = k[1:-1]              # Elongation constants
    k_termination = k[-1]               # Termination rate
    gene_Length = len(k)-2               # Number of codons removed the initiation and termination rates
    t = t_array[0]                      # Initial time
    max_ribosome_id = -1                # Counter for assigning unique IDs

    ## Initialize variables
    number_TimePoints = len(t_array)
    t_final = t_array[-1]
    occupancy_output = np.zeros((gene_Length+2, number_TimePoints))  # Preallocate occupancy matrix adding the start and stop codons
    iteration = 0

    ribosomes = {}  # Dictionary to store active ribosomes: {ribosome_id: {'position': position, 'initiation_time': initiation_time}}
    ribosome_positions_list = []  # List to store position arrays for each ribosome

    ## Run SSA
    while t < t_final:
        ## Handle perturbations
        if t >= timePerturbationApplication and evaluatingInhibitor == 1:
            Inhibitor_Condition = 0  # Inhibitor is active
        else:
            Inhibitor_Condition = 1  # Inhibitor is inactive
        if evaluatingFRAP == 1 and timePerturbationApplication <= t <= timePerturbationApplication + 10:
            ribosomes = {}  # Reset ribosome positions
            # Mark all existing ribosomes as terminated by filling remaining positions with NaN
            for positions_array in ribosome_positions_list:
                positions_array[iteration:] = np.nan

        ## Compute propensity functions
        Nribosomes = len(ribosomes)  # Number of active ribosomes

        # Initiation propensity
        initiation_possible = False
        if Nribosomes == 0:
            initiation_possible = True
        else:
            # Check exclusion at start
            positions = np.array([ribo['position'] for ribo in ribosomes.values()])
            if np.min(positions) > exclusion:
                initiation_possible = True
        initiation_propensity = k_bind * Inhibitor_Condition if initiation_possible else 0

        # Elongation propensities
        elongation_propensities = []
        elongation_ribosome_ids = []
        ribosome_positions = ribosomes.copy()
        for ribosome_id, ribo in ribosome_positions.items():
            position = ribo['position']
            if position >= gene_Length:
                continue  # Ribosome at or beyond the end, termination will handle it
            else:
                # Check for ribosomes ahead within exclusion distance
                positions_ahead = np.array([r['position'] for r in ribosomes.values() if r['position'] > position])
                min_ahead_position = np.min(positions_ahead) if len(positions_ahead) > 0 else np.inf
                if (position + exclusion) < min_ahead_position:
                    propensity = k_elongation[int(position) - 1]  # k_elongation uses zero-based indexing
                    elongation_propensities.append(propensity)
                    elongation_ribosome_ids.append(ribosome_id)
                else:
                    # Cannot elongate due to exclusion
                    pass

        # Termination propensities
        termination_propensities = []
        termination_ribosome_ids = []
        for ribosome_id, ribo in ribosomes.items():
            position = ribo['position']
            if position >= gene_Length:
                termination_propensities.append(k_termination)
                termination_ribosome_ids.append(ribosome_id)

        # Build propensities and reactions
        propensities = []
        reactions = []

        # Initiation
        if initiation_propensity > 0:
            propensities.append(initiation_propensity)
            reactions.append(('initiation', None))

        # Elongation
        for propensity, ribosome_id in zip(elongation_propensities, elongation_ribosome_ids):
            propensities.append(propensity)
            reactions.append(('elongation', ribosome_id))

        # Termination
        for propensity, ribosome_id in zip(termination_propensities, termination_ribosome_ids):
            propensities.append(propensity)
            reactions.append(('termination', ribosome_id))

        propensities = np.array(propensities)
        sum_propensities = np.sum(propensities)

        ## Update time
        if sum_propensities == 0:
            # No reactions can occur; advance time to final
            t = t_final
        else:
            tau = -np.log(np.random.rand()) / sum_propensities
            if evaluatingInhibitor == 1 and t < timePerturbationApplication and (t + tau) > timePerturbationApplication:
                t = timePerturbationApplication
            else:
                t += tau
                ## Select reaction
                r2 = sum_propensities * np.random.rand()
                cumulative_propensity = np.cumsum(propensities)
                reaction_index = np.searchsorted(cumulative_propensity, r2)
                reaction, ribosome_id = reactions[reaction_index]
                ## Update state
                if reaction == 'initiation':
                    # Initiation
                    max_ribosome_id += 1
                    ribosome_id = max_ribosome_id
                    ribosomes[ribosome_id] = {'position': 1, 'initiation_time': t}
                    # Create position array for this ribosome
                    positions_array = np.full(number_TimePoints, np.nan)
                    ribosome_positions_list.append(positions_array)
                elif reaction == 'elongation':
                    # Elongation
                    ribosomes[ribosome_id]['position'] += 1
                elif reaction == 'termination':
                    # Termination
                    del ribosomes[ribosome_id]  # Remove ribosome from active list

        ## Generate output
        while iteration < number_TimePoints and t > t_array[iteration]:
            # Compute occupancy vector
            occupancy_vector = np.zeros(gene_Length)
            for ribo in ribosomes.values():
                position = ribo['position']
                if 1 <= position <= gene_Length:
                    occupancy_vector[int(position) - 1] = 1  
            occupancy_output[1:-1, iteration] = occupancy_vector
            # Update positions of ribosomes
            for idx, positions_array in enumerate(ribosome_positions_list):
                ribosome_id = idx
                if ribosome_id in ribosomes:
                    initiation_time = ribosomes[ribosome_id]['initiation_time']
                    if t_array[iteration] >= initiation_time:
                        # Ribosome is active
                        positions_array[iteration] = ribosomes[ribosome_id]['position']
                    else:
                        # Ribosome not yet initiated
                        positions_array[iteration] = np.nan
                else:
                    # Ribosome is not active (terminated)
                    positions_array[iteration] = np.nan
            iteration += 1

    # Convert ribosome_positions_list to numpy array
    ribosome_trajectories = np.array(ribosome_positions_list)
    ribosome_trajectories[np.isnan(ribosome_trajectories)] = 0
    ribosome_trajectories = ribosome_trajectories.astype(int)

    return ribosome_trajectories, occupancy_output

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define parameters for a gene with 1000 codons
k_bind = 0.05  # Initiation rate
k_elongation = np.ones(1800) * 10 # Elongation rates for each codon
k_termination = 1  # Termination rate
k = np.concatenate(([k_bind], k_elongation, [k_termination]))
gene_length = len(k)
# Define time array from 0 to 100 seconds, recording every second
t_array = np.arange(0, 1001, 1)

# Run the simulation
ribosome_trajectories, occupancy_output = TASEP_SSA(k, t_array)

print(ribosome_trajectories.shape, occupancy_output.shape)


In [ ]:
ribosome_trajectories.shape

In [ ]:
# number_repetitons = 1
# list_ribosome_trajectories = []
# list_occupancy_output = []
# for i in range(number_repetitons):
#     ribosome_trajectories, occupancy_output = TASEP_SSA(k, t_array)
#     list_ribosome_trajectories.append(ribosome_trajectories)
#     list_occupancy_output.append(occupancy_output)




In [ ]:
plt.plot(t_array, ribosome_trajectories[1,:], label=f'Ribosome {0}')


In [ ]:
for i in range(ribosome_trajectories.shape[0]):
    plt.plot(t_array, ribosome_trajectories[i,:], label=f'Ribosome {i}')


In [ ]:
plt.figure(figsize=(10, 5), facecolor='black')
plt.matshow(occupancy_output.T, aspect='auto', cmap='binary_r')
plt.xlabel('Codon')
plt.ylabel('Time')
plt.title('kymograph')
plt.grid(False)
plt.show()   

In [ ]:
# Calculate the number of ribosomes over time
Nribosomes_over_time = np.sum(ribosome_trajectories > 0, axis=0)

# Plot the number of ribosomes over time
plt.figure(figsize=(8, 4))
plt.plot(t_array, Nribosomes_over_time, label='Number of Ribosomes')
#plt.axvline(x=timePerturbationApplication, color='r', linestyle='--', label='Inhibitor Applied')
plt.xlabel('Time (s)')
plt.ylabel('Number of Ribosomes')
plt.title('Ribosome Dynamics over Time')
plt.legend()
#plt.grid()
plt.show()

In [ ]:
# converting ribosome positions to intensity
# linspace from 10 to 100 with 10 steps
tag_positions_first_probe_vector = np.arange(10, 101, 10)
tag_positions_second_probe_vector = np.linspace(500, 1800, 6, endpoint=False).astype(int) # [500] 
first_probe_position_vector = np.zeros(gene_length)

second_probe_position_vector = np.zeros(gene_length)    

for tag in tag_positions_first_probe_vector:
    first_probe_position_vector[tag:] += 1  

for tag in tag_positions_second_probe_vector:
    second_probe_position_vector[tag:] += 1

intensity_vector_first_signal = np.sum( first_probe_position_vector * occupancy_output.T, axis=1)
intensity_vector_second_signal = np.sum( second_probe_position_vector * occupancy_output.T, axis=1)


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(intensity_vector_first_signal/np.max(intensity_vector_first_signal), label='First Signal')
plt.plot(intensity_vector_second_signal/np.max(intensity_vector_second_signal), label='Second Signal')
plt.xlabel('Time (s)')
plt.ylabel('Intensity (ump)')
plt.title('Intensity over Time')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

def plot_RibosomeMovement(RibosomePositions, IntensityVector, probePositions, time, geneLength, SecondIntensityVector=None, second_probePositions=None, fileNameGif='temp_gif', color='cyan',second_color ='deeppink', FrameVelocity=10, timePerturbationApplication= None):
    """
    Function to plot ribosome movement and intensity over time, and generate an animation as a GIF.

    Parameters:
    - RibosomePositions: numpy array of shape (num_ribosomes, num_timepoints)
    - IntensityVector: numpy array of length num_timepoints
    - time: numpy array of time points
    - geneLength: length of the gene (scalar)
    - fileNameGif: filename for the output GIF (without extension)
    - probePositions: numpy array of probe positions along the gene
    - timePerturbationApplication: time when perturbation is applied
    - color: color to use for plotting (e.g., 'blue')
    - FrameVelocity: frames per second (int)
    """
    # Normalize IntensityVector
    IntensityVector = IntensityVector / np.max(IntensityVector)
    if SecondIntensityVector is not None:
        SecondIntensityVector = SecondIntensityVector / np.max(SecondIntensityVector)
    maxIntensity = 1
    Max_No_Ribosomes, num_timepoints = RibosomePositions.shape


    timePoints = len(time)
    if geneLength > 1100:
        pointSize = 4.5
    else:
        pointSize = 6

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 3.5), facecolor='black',gridspec_kw={'height_ratios': [0.6, 0.4]})
    fig.subplots_adjust(hspace=0.5)
    stepSize = 5

    # Prepare the frames for animation
    frames = range(0, timePoints, stepSize)

    # Initialize plots
    def init():
        # Upper plot (Intensity over time)
        ax1.set_facecolor('black')
        ax1.set_xlim(0, time[-1])
        ax1.set_ylim(0, maxIntensity * 1.2)
        ax1.set_xticks([])
        ax1.set_yticks([])
        ax1.set_xlabel(f'Time', fontsize=10, color='white')
        ax1.set_ylabel('Intensity', fontsize=10, color='white')
        ax1.grid(False)
        
        # Lower plot (Ribosome movement)
        ax2.set_facecolor('black')
        ax2.set_xlim(0, geneLength + 1)
        ax2.set_ylim(0.09, 0.15)
        ax2.axis('off')
        # grid off
        ax2.grid(False)
        return []

    # Animation function
    def animate(frame_idx):
        tp = frame_idx
        ax1.clear()
        ax2.clear()

        # Plot settings for upper plot
        ax1.set_facecolor('black')
        ax1.set_xlim(0, time[-1])
        ax1.set_ylim(0, maxIntensity * 1.2)
        ax1.set_xlabel(f'Time', fontsize=10, color='white')
        ax1.set_ylabel('Intensity', fontsize=10, color='white')
        ax1.plot([0, time[-1]], [0, 0], '-w', linewidth=2)
        ax1.plot([0, 0], [0, maxIntensity * 1.1], '-w', linewidth=2)
        # add white axis ticks
        ax1.tick_params(axis='x', colors='white')
        ax1.tick_params(axis='y', colors='white')

        # Plot intensity
        if IntensityVector[tp] > 0 :
            ax1.plot(time[tp], IntensityVector[tp], 'o', markersize=5,
                    markeredgecolor=color, markerfacecolor=color)
        ax1.plot(time[:tp], IntensityVector[:tp], '-', color=color, linewidth=2)

        if SecondIntensityVector is not None:
            if SecondIntensityVector[tp] > 0 :
                ax1.plot(time[tp], SecondIntensityVector[tp], 's', markersize=5,
                        markeredgecolor=second_color, markerfacecolor=second_color)
            ax1.plot(time[:tp], SecondIntensityVector[:tp], '-', color=second_color, linewidth=2)


        # Plot perturbation line and label
        if timePerturbationApplication is not None:
            if time[tp] >= timePerturbationApplication:
                ax1.text(5, maxIntensity * 1.3, 'Harringtonine', color='red', fontsize=8)
                ax1.plot([timePerturbationApplication, timePerturbationApplication],
                         [0, maxIntensity * 1.3], 'r-', linewidth=1)

        # Add title on the first frame
        #if tp == 0:
        ax1.text(time[-1] / 2.3, maxIntensity * 1.4, 'Ribosome Movement',
                color='white', fontsize=14)
        ax1.grid(False)
        # Plot settings for lower plot
        ax2.set_facecolor('black')
        ax2.set_xlim(0, geneLength + 1)
        ax2.set_ylim(0.0, 0.15)
        ax2.axis('off')


        # Plot gene line and probes
        ax2.plot([0, geneLength], [0.1, 0.1], 'w-', linewidth=2)
        ax2.plot(probePositions, np.full_like(probePositions, 0.5), 's',
                 markersize=5, markeredgecolor=color, markerfacecolor=color)

        # Plot ribosomes
        for i in range(Max_No_Ribosomes):
            position = RibosomePositions[i, tp]
            if position > 0 and position <= geneLength:
                # Ribosome body
                ax2.plot(position, 0.1, 'o', markersize=15,
                         markeredgecolor=[0.5, 0.5, 0.5],
                         markerfacecolor=[0.7, 0.7, 0.7])
                # Ribosome activity indicator
                numberOfProbesPassed = np.sum(probePositions <= position) / len(probePositions)
                markerSize = 0.1 + 4 * numberOfProbesPassed
                ax2.plot(position, 0.102, 'o', markersize=markerSize,
                         markeredgecolor=color, markerfacecolor=color)
                # activity indicator for second probe
                if second_probePositions is not None:
                    numberOfProbesPassed = np.sum(second_probePositions <= position) / len(second_probePositions)
                    markerSize = 0.1 + 2 * numberOfProbesPassed
                    ax2.plot(position, 0.104, 'o', markersize=markerSize,
                             markeredgecolor=second_color, markerfacecolor=second_color)

        # Time label
        time_str = f'{time[tp]:.0f} s'
        ax2.text(geneLength + 10, 0.1, time_str, color='white', fontsize=8)

        return []

    ani = FuncAnimation(fig, animate, frames=frames, init_func=init, blit=False,
                        interval=1000 / FrameVelocity)

    # Save animation as GIF
    writergif = PillowWriter(fps=FrameVelocity)
    ani.save(f'{fileNameGif}.gif', writer=writergif)

    display(IPImage(filename= f'{fileNameGif}.gif'   ))

    plt.close(fig)

In [ ]:
# Call the function
plot_RibosomeMovement(ribosome_trajectories, intensity_vector_first_signal ,first_probe_position_vector, t_array, gene_length,SecondIntensityVector=intensity_vector_second_signal,second_probePositions=second_probe_position_vector) # intensity_vector_second_signal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib import gridspec


def plot_spot(amplitude, sigma=2, grid_size=13):
    mu_x, mu_y = (grid_size/2)-0.5, (grid_size/2)-0.5
    x = np.linspace(0, grid_size - 1, grid_size)
    y = np.linspace(0, grid_size - 1, grid_size)
    x, y = np.meshgrid(x, y)
    z = amplitude * np.exp(-((x - mu_x)**2 + (y - mu_y)**2) / (2 * sigma**2))
    # Normalize and return as uint8
    if z.max() > 0:
        z = (255*(z/z.max())).astype(np.uint8)
    else:
        z = z.astype(np.uint8)
    return z

def plot_RibosomeMovement_and_Microscope(RibosomePositions, IntensityVector, probePositions, time, geneLength, SecondIntensityVector=None, second_probePositions=None, fileNameGif='temp_gif', color='cyan', second_color='deeppink', FrameVelocity=10, timePerturbationApplication=None):
    # Normalize IntensityVector
    IntensityVector = IntensityVector / np.max(IntensityVector)
    if SecondIntensityVector is not None:
        SecondIntensityVector = SecondIntensityVector / np.max(SecondIntensityVector)
    maxIntensity = 1
    Max_No_Ribosomes, num_timepoints = RibosomePositions.shape

    timePoints = len(time)
    if geneLength > 1100:
        pointSize = 4.5
    else:
        pointSize = 6

    # Create figure with a specific size
    fig = plt.figure(figsize=(12, 6), facecolor='black')

    # Create a gridspec layout within the figure
    gs = gridspec.GridSpec(2, 3, height_ratios=[0.3, 0.7], width_ratios=[2, 0.5, 0.5])

    # Create subplots
    ax1 = fig.add_subplot(gs[0, 0])  # First row, first column
    ax2 = fig.add_subplot(gs[1, 0])  # Second row, first column
    ax3 = fig.add_subplot(gs[:, 1])  # Both rows, second column (merged vertically)
    ax4 = fig.add_subplot(gs[:, 2])  # Both rows, third column (merged vertically)

    normalized_intensity_vector_first_signal = IntensityVector / np.max(IntensityVector)
    if SecondIntensityVector is not None:
        normalized_intensity_vector_second_signal = SecondIntensityVector / np.max(SecondIntensityVector)

    stepSize = 5

    # Prepare the frames for animation
    frames = range(0, timePoints, stepSize)

    # Initialize plots
    def init():
        # Upper plot (Intensity over time)
        ax1.set_facecolor('black')
        ax1.set_xlim(0, time[-1])
        ax1.set_ylim(0, maxIntensity * 1.2)
        ax1.set_xticks([])
        ax1.set_yticks([])
        ax1.set_xlabel('Time', fontsize=10, color='white')
        ax1.set_ylabel('Intensity', fontsize=10, color='white')
        ax1.grid(False)
        
        # Lower plot (Ribosome movement)
        ax2.set_facecolor('black')
        ax2.set_xlim(0, geneLength + 1)
        ax2.set_ylim(0.0, 0.15)
        ax2.axis('off')
        ax2.grid(False)

        # Microscope image axes
        ax3.set_facecolor('black')
        ax3.set_xticks([])
        ax3.set_yticks([])
        ax3.grid(False)
        ax3.axis('off')

        ax4.set_facecolor('black')
        ax4.set_xticks([])
        ax4.set_yticks([])
        ax4.grid(False)
        ax4.axis('off')
        
        return []

    # Animation function
    def animate(frame_idx):
        tp = frame_idx
        ax1.clear()
        ax2.clear()
        ax3.clear()
        ax4.clear()

        # Plot settings for upper plot
        ax1.set_facecolor('black')
        ax1.set_xlim(0, time[-1])
        ax1.set_ylim(0, maxIntensity * 1.2)
        ax1.set_xlabel('Time', fontsize=10, color='white')
        ax1.set_ylabel('Intensity', fontsize=10, color='white')
        ax1.plot([0, time[-1]], [0, 0], '-w', linewidth=2)
        ax1.plot([0, 0], [0, maxIntensity * 1.1], '-w', linewidth=2)
        ax1.tick_params(axis='x', colors='white')
        ax1.tick_params(axis='y', colors='white')

        # Plot intensity
        if IntensityVector[tp] > 0:
            ax1.plot(time[tp], IntensityVector[tp], 'o', markersize=5,
                     markeredgecolor=color, markerfacecolor=color)
        ax1.plot(time[:tp+1], IntensityVector[:tp+1], '-', color=color, linewidth=2)

        if SecondIntensityVector is not None:
            if SecondIntensityVector[tp] > 0:
                ax1.plot(time[tp], SecondIntensityVector[tp], 's', markersize=5,
                         markeredgecolor=second_color, markerfacecolor=second_color)
            ax1.plot(time[:tp+1], SecondIntensityVector[:tp+1], '-', color=second_color, linewidth=2)

        # Plot perturbation line and label
        if timePerturbationApplication is not None:
            if time[tp] >= timePerturbationApplication:
                ax1.text(5, maxIntensity * 1.3, 'Harringtonine', color='red', fontsize=8)
                ax1.plot([timePerturbationApplication, timePerturbationApplication],
                         [0, maxIntensity * 1.3], 'r-', linewidth=1)

        # Add title
        ax1.text(time[-1] / 2.3, maxIntensity * 1.4, 'Ribosome Movement',
                 color='white', fontsize=14)
        ax1.grid(False)

        # Plot settings for lower plot
        ax2.set_facecolor('black')
        ax2.set_xlim(0, geneLength + 1)
        ax2.set_ylim(0.0, 0.15)
        ax2.axis('off')

        # Plot gene line and probes
        ax2.plot([0, geneLength], [0.1, 0.1], 'w-', linewidth=2)
        ax2.plot(probePositions, np.full_like(probePositions, 0.1), 's',
                 markersize=5, markeredgecolor=color, markerfacecolor=color)

        # Plot ribosomes
        for i in range(Max_No_Ribosomes):
            position = RibosomePositions[i, tp]
            if position > 0 and position <= geneLength:
                # Ribosome body
                ax2.plot(position, 0.1, 'o', markersize=15,
                         markeredgecolor=[0.5, 0.5, 0.5],
                         markerfacecolor=[0.7, 0.7, 0.7])
                # Ribosome activity indicator
                numberOfProbesPassed = np.sum(probePositions <= position) / len(probePositions)
                markerSize = 0.1 + 4 * numberOfProbesPassed
                ax2.plot(position, 0.102, 'o', markersize=markerSize,
                         markeredgecolor=color, markerfacecolor=color)
                # Activity indicator for second probe
                if second_probePositions is not None:
                    numberOfProbesPassed2 = np.sum(second_probePositions <= position) / len(second_probePositions)
                    markerSize2 = 0.1 + 2 * numberOfProbesPassed2
                    ax2.plot(position, 0.104, 'o', markersize=markerSize2,
                             markeredgecolor=second_color, markerfacecolor=second_color)

        # Time label
        time_str = f'{time[tp]:.0f} s'
        ax2.text(geneLength + 10, 0.1, time_str, color='white', fontsize=8)

        # Plot microscope images on ax3 and ax4
        amplitude = normalized_intensity_vector_first_signal[tp]
        noise_percentage = 0.05
        max_noise_size = int(255 * noise_percentage)
        sigma = 1 + amplitude * 2
        z = plot_spot(amplitude, sigma=sigma)
        added_noise = np.random.normal(0, max_noise_size, z.shape)
        z = z + added_noise
        z = np.clip(z, 0, 255)

        ax3.imshow(z, cmap='gray', vmax=255)
        ax3.set_title('Channel 0', color='white')
        ax3.set_xticks([])
        ax3.set_yticks([])
        for spine in ax3.spines.values():
            spine.set_color('white')
        ax3.set_facecolor('black')
        ax3.set_aspect('equal')

        if SecondIntensityVector is not None:
            amplitude2 = SecondIntensityVector[tp]
            sigma2 = 1 + amplitude2 * 3
            z2 = plot_spot(amplitude2, sigma=sigma2)
            added_noise2 = np.random.normal(0, max_noise_size, z2.shape)
            z2 = z2 + added_noise2
            z2 = np.clip(z2, 0, 255)
            ax4.imshow(z2, cmap='gray', vmax=255)
            ax4.set_title('Channel 1', color='white')
            ax4.set_xticks([])
            ax4.set_yticks([])
            for spine in ax4.spines.values():
                spine.set_color('white')
            ax4.set_facecolor('black')
            ax4.set_aspect('equal')
        else:
            ax4.axis('off')

        return []

    ani = FuncAnimation(fig, animate, frames=frames, init_func=init, blit=False,
                        interval=1000 / FrameVelocity)

    # Save animation as GIF
    ani.save(f'{fileNameGif}.gif', writer=PillowWriter(fps=FrameVelocity))
    display(IPImage(filename= f'{fileNameGif}.gif'   ))
    plt.close(fig)

In [ ]:
plot_RibosomeMovement_and_Microscope(ribosome_trajectories, intensity_vector_first_signal ,first_probe_position_vector, t_array, gene_length,SecondIntensityVector=intensity_vector_second_signal,second_probePositions=second_probe_position_vector) # intensity_vector_second_signal

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

def TASEP_ODE(p, t, k_bind, k_elongation, k_termination):
    N = len(k_elongation) + 1  # Total number of codon positions
    dpdt = np.zeros(N)
    # Position 1
    dpdt[0] = k_bind - k_elongation[0] * p[0]
    # Positions 2 to N-1
    for i in range(1, N - 1):
        dpdt[i] = k_elongation[i - 1] * p[i - 1] - k_elongation[i] * p[i]
    # Position N
    dpdt[N - 1] = k_elongation[N - 2] * p[N - 2] - k_termination * p[N - 1]
    return dpdt

# Parameters
k_bind = 0.05  # Initiation rate
gene_length = 1800+2  # Number of codons
k_elongation = np.full(gene_length - 1, 10)  # Elongation rates for positions 1 to N-1
k_termination = k_elongation[-1]  # Termination rate

# Initial conditions
N = gene_length
p0 = np.zeros(N)  # Start with zero concentration at all positions
# Time points
t = np.linspace(0, 1000, 1000)
# Solve ODEs
p = odeint(TASEP_ODE, p0, t, args=(k_bind, k_elongation, k_termination))

# Calculate the total number of ribosomes over time
Nribosomes_over_time = np.sum(p, axis=1)


Nribosomes_over_time_ssa = np.sum(ribosome_trajectories > 0, axis=0)
# Plot total number of ribosomes over time
plt.figure(figsize=(10, 6))
plt.plot(t, Nribosomes_over_time, label='ODE Ribosome Count')
plt.plot(t_array, Nribosomes_over_time_ssa, label='SSA Ribosome Count')
plt.xlabel('Time')
plt.ylabel('Ribosome Count')
plt.title('Time')
plt.legend()
plt.show()




In [ ]:
# Calculate intensity vectors using deterministic concentrations
intensity_vector_first_signal_ode = np.dot(first_probe_position_vector, p.T)
intensity_vector_second_signal_ode = np.dot(second_probe_position_vector, p.T)

# Plotting intensity signals over time
plt.figure(figsize=(12, 6))
plt.plot(t, intensity_vector_first_signal_ode/np.max(intensity_vector_first_signal_ode), label='First  Intensity ODE',color='r')
plt.plot(t, intensity_vector_second_signal_ode/np.max(intensity_vector_second_signal_ode), label='Second  Intensity ODE',color='b')
plt.plot(intensity_vector_first_signal/np.percentile(intensity_vector_first_signal,80), label='First Intensity SSA',color='r')
plt.plot(intensity_vector_second_signal/np.percentile(intensity_vector_second_signal,80), label='Second Intensity SSA',color='b')
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.title('Intensity Over Time from Deterministic Model')
plt.legend()
plt.show()